# [9665] Spam Classification
Data file
* https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/SMS_spam.tsv

In [ ]:
from datetime import datetime
print(f'Run time: {datetime.now().strftime("%D %T")}')

Run time: 02/09/25 13:20:54


### Import libraries

In [ ]:
import pandas as pd
import string
import re
import nltk
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [ ]:
# Details at https://www.nltk.org/data.html
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

### Load data

In [ ]:
# Read SMS_spam.tsv into dataframe
#  Note: Field separator is '\t'
df = pd.read_csv('https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/SMS_spam.tsv',
                 sep='\t', names=['label', 'body_text'], header=None)
df.shape

(5568, 2)

In [ ]:
pd.set_option('max_colwidth',200)

In [ ]:
df.head()

,label,body_text
0,ham,I've been searching for the right words to thank you for this breather. I promise i wont take your help for granted and will fulfil my promise. You have been wonderful and a blessing at all times.
1,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
2,ham,"Nah I don't think he goes to usf, he lives around here though"
3,ham,Even my brother is not like to speak with me. They treat me like aids patent.
4,ham,I HAVE A DATE ON SUNDAY WITH WILL!!


### Examine data

In [ ]:
# Review class distribution
df['label'].value_counts()

,count
label,
ham,4822
spam,746


In [ ]:
# Check for missing values
df.isnull().sum()

,0
label,0
body_text,0


### Preprocess data

In [ ]:
# Define stop words list
stopwords = nltk.corpus.stopwords.words('english')     # All English Stopwords
print(stopwords)

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

In [ ]:
# Instantiate Porter Stemmer
ps = nltk.PorterStemmer()

#### Create functions to lowercase, remove punctuation, tokenize, remove stopwords, and stem

In [ ]:
# Function clean_text will be used in subsequent cells
def clean_text(text):
    text = "".join([word.lower() for word in text if word not in string.punctuation])
    tokens = re.split('\W+', text)
    text = [ps.stem(word) for word in tokens if word not in stopwords]
    return text     # Returns a list of strings

In [ ]:
# Function clean_text_2 will be used in subsequent cells
def clean_text_2(text):
    text = "".join([word.lower() for word in text if word not in string.punctuation])
    tokens = re.split('\W+', text)
    text = [ps.stem(word) for word in tokens if word not in stopwords]
    text_2 = ' '.join(word for word in text)
    return text_2     # Returns one string

### Vectorize data - Bag of Words (BoW)

#### CountVectorizer converts a collection of text documents into a matrix of token counts

In [ ]:
# Apply CountVectorizer
bow_vect = CountVectorizer(analyzer=clean_text)
bow_counts = bow_vect.fit_transform(df['body_text'])
print(bow_counts.shape)
print()
print(bow_vect.get_feature_names_out())

(5568, 8107)

['' '0' '008704050406' ... 'ü' 'üll' '〨ud']


In [ ]:
bow_counts

<5568x8107 sparse matrix of type '<class 'numpy.int64'>'
	with 50137 stored elements in Compressed Sparse Row format>

In [ ]:
bow_counts_df = pd.DataFrame(bow_counts.toarray(), columns=bow_vect.get_feature_names_out())
bow_counts_df.head()

,,0,008704050406,0089mi,0121,01223585236,01223585334,0125698789,02,020603,...,zindgi,zoe,zogtoriu,zoom,zouk,zyada,é,ü,üll,〨ud
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Vectorize data - N-grams

In [ ]:
df['body_text_clean'] = df['body_text'].apply(clean_text_2)
df.head()

,label,body_text,body_text_clean
0,ham,I've been searching for the right words to thank you for this breather. I promise i wont take your help for granted and will fulfil my promise. You have been wonderful and a blessing at all times.,ive search right word thank breather promis wont take help grant fulfil promis wonder bless time
1,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,free entri 2 wkli comp win fa cup final tkt 21st may 2005 text fa 87121 receiv entri questionstd txt ratetc appli 08452810075over18
2,ham,"Nah I don't think he goes to usf, he lives around here though",nah dont think goe usf live around though
3,ham,Even my brother is not like to speak with me. They treat me like aids patent.,even brother like speak treat like aid patent
4,ham,I HAVE A DATE ON SUNDAY WITH WILL!!,date sunday


In [ ]:
 # Apply only bigram vectorizer: ngram_range=(2,2)
ngram_vect = CountVectorizer(ngram_range=(2,2))
ngram_counts = ngram_vect.fit_transform(df['body_text_clean'])
print(ngram_counts.shape)
print()
print(ngram_vect.get_feature_names_out())

(5568, 31275)

['008704050406 sp' '0089mi last' '0121 2025050' ... 'üll submit'
 'üll take' '〨ud even']


In [ ]:
ngram_counts

<5568x31275 sparse matrix of type '<class 'numpy.int64'>'
	with 43729 stored elements in Compressed Sparse Row format>

In [ ]:
ngram_counts_df = pd.DataFrame(ngram_counts.toarray(), columns=ngram_vect.get_feature_names_out())
ngram_counts_df.head()

,008704050406 sp,0089mi last,0121 2025050,01223585236 xx,01223585334 cum,0125698789 ring,02 user,020603 2nd,0207 153,02072069400 bx,...,zoe 18,zoe hit,zogtoriu stare,zoom cine,zouk nichol,zyada kisi,üll finish,üll submit,üll take,〨ud even
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Vectorize data - TF-IDF

In [ ]:
# Apply TfidfVectorizer
tfidf_vect = TfidfVectorizer(analyzer=clean_text)
tfidf_counts = tfidf_vect.fit_transform(df['body_text'])
print(tfidf_counts.shape)
print()
print(tfidf_vect.get_feature_names_out())

(5568, 8107)

['' '0' '008704050406' ... 'ü' 'üll' '〨ud']


In [ ]:
tfidf_counts

<5568x8107 sparse matrix of type '<class 'numpy.float64'>'
	with 50137 stored elements in Compressed Sparse Row format>

In [ ]:
tfidf_counts_df = pd.DataFrame(tfidf_counts.toarray(), columns=tfidf_vect.get_feature_names_out())
tfidf_counts_df.head()

,,0,008704050406,0089mi,0121,01223585236,01223585334,0125698789,02,020603,...,zindgi,zoe,zogtoriu,zoom,zouk,zyada,é,ü,üll,〨ud
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Define global variables

In [ ]:
global TEST_SIZE, RANDOM_STATE
TEST_SIZE = .3
RANDOM_STATE = 42

### Train random forest model (using Bag of Words)

In [ ]:
# Prepare data set for model training
bow_counts_df['label'] = df['label']
bow_counts_df.head()

,,0,008704050406,0089mi,0121,01223585236,01223585334,0125698789,02,020603,...,zoe,zogtoriu,zoom,zouk,zyada,é,ü,üll,〨ud,label
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ham
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,spam
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ham
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ham
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ham


In [ ]:
# Separate independent and dependent variables
X = bow_counts_df.drop('label', axis=1)
y = bow_counts_df['label']

In [ ]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE,
                                                    random_state=RANDOM_STATE)

In [ ]:
# Instantiate random forest classifier
rf = RandomForestClassifier(random_state=RANDOM_STATE)

In [ ]:
%%time

# Train random forest classifier model on training data
rf.fit(X_train, y_train);

CPU times: user 20.9 s, sys: 128 ms, total: 21 s
Wall time: 22.6 s


RandomForestClassifier(random_state=42)

In [ ]:
# Make predictions for the test set
y_pred_test = rf.predict(X_test)

# View accuracy score
print(f"Accuracy using BoW: {round(accuracy_score(y_test, y_pred_test)*100,3)}%")

Accuracy using BoW: 97.606%


### Train random forest model (using N-grams)

In [ ]:
# Prepare data set for model training
ngram_counts_df['label'] = df['label']
ngram_counts_df.head()

,008704050406 sp,0089mi last,0121 2025050,01223585236 xx,01223585334 cum,0125698789 ring,02 user,020603 2nd,0207 153,02072069400 bx,...,zoe hit,zogtoriu stare,zoom cine,zouk nichol,zyada kisi,üll finish,üll submit,üll take,〨ud even,label
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ham
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,spam
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ham
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ham
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ham


In [ ]:
# Separate independent and dependent variables
X = ngram_counts_df.drop('label', axis=1)
y = ngram_counts_df['label']

In [ ]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE,
                                                    random_state=RANDOM_STATE)

In [ ]:
%%time

# Train random forest classifier model on training data
rf.fit(X_train, y_train);

CPU times: user 2min 17s, sys: 317 ms, total: 2min 17s
Wall time: 2min 18s


RandomForestClassifier(random_state=42)

In [ ]:
# Make predictions for the test set
y_pred_test = rf.predict(X_test)

# View accuracy score
print(f"Accuracy using N-grams: {round(accuracy_score(y_test, y_pred_test)*100,3)}%")

Accuracy using N-grams: 95.272%


### Train random forest model (using TF-IDF)

In [ ]:
# Prepare data set for model training
tfidf_counts_df['label'] = df['label']
tfidf_counts_df.head()

,,0,008704050406,0089mi,0121,01223585236,01223585334,0125698789,02,020603,...,zoe,zogtoriu,zoom,zouk,zyada,é,ü,üll,〨ud,label
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ham
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,spam
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ham
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ham
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ham


In [ ]:
# Separate independent and dependent variables
X = tfidf_counts_df.drop('label', axis=1)
y = tfidf_counts_df['label']

In [ ]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE,
                                                    random_state=RANDOM_STATE)

In [ ]:
%%time

# Train random forest classifier model on training data
rf.fit(X_train, y_train);

CPU times: user 21.4 s, sys: 52.3 ms, total: 21.4 s
Wall time: 21.4 s


RandomForestClassifier(random_state=42)

In [ ]:
# Make predictions for the test set
y_pred_test = rf.predict(X_test)

# View accuracy score
print(f"Accuracy using TF-IDF: {round(accuracy_score(y_test, y_pred_test)*100,3)}%")

Accuracy using TF-IDF: 97.666%
